# 🐍 Fundamentos de Python para el SOC

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/florvela/IA-y-automatizacion-en-seguridad-defensiva/blob/main/codigos-de-ejemplo/clase_en_vivo/live_01_python.ipynb)

**Presentar entre las diapositivas 40 y 41** (después de "Veamos un ejemplo").

En este notebook vemos, en formato corto:
1. Parsear un log con **regex**
2. **Credenciales** seguras (nunca hardcodeadas)
3. **Logging** en vez de `print()`
4. **Resiliencia** con un decorador de reintentos

<img src="https://raw.githubusercontent.com/florvela/IA-y-automatizacion-en-seguridad-defensiva/main/02-fundamentos-python/images/dos_and_donts.png" width="460"/>


## 1. Parsear un log con regex
De una línea de syslog sacamos fecha, host, proceso y mensaje.

In [ ]:
import re

log = "Nov 12 08:15:03 web-01 sshd[2451]: Failed password for admin from 10.0.0.5 port 4522"

patron = r"^(?P<fecha>\w+\s+\d+\s[\d:]+)\s(?P<host>\S+)\s(?P<proceso>\w+)"
m = re.match(patron, log)

evento = m.groupdict()
evento["ip"] = re.search(r"\d+\.\d+\.\d+\.\d+", log).group()
print(evento)

## 2. Credenciales seguras
La API key **nunca** va en el código: la leemos de una variable de entorno.

<img src="https://raw.githubusercontent.com/florvela/IA-y-automatizacion-en-seguridad-defensiva/main/02-fundamentos-python/images/credenciales.png" width="420"/>

In [ ]:
import os

# ❌ MAL:  API_KEY = "sk-1234-super-secreta"
# ✅ BIEN: la leemos del entorno (o de un archivo .env que NO se sube a git)
API_KEY = os.getenv("VT_API_KEY", "(no configurada)")
print("Usando API key:", API_KEY[:6], "...")

## 3. Logging en vez de print()
En producción el script corre solo, de madrugada. `logging` nos da timestamp y nivel.

<img src="https://raw.githubusercontent.com/florvela/IA-y-automatizacion-en-seguridad-defensiva/main/02-fundamentos-python/images/logging.png" width="420"/>

In [ ]:
import logging

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("soc")

log.info("Analizando alerta...")
log.warning("IP 10.0.0.5 con 5 intentos fallidos")
log.error("No se pudo conectar a VirusTotal")

## 4. Resiliencia: reintentar si la API falla
Un decorador envuelve la función y la reintenta sin cambiar su código.

<img src="https://raw.githubusercontent.com/florvela/IA-y-automatizacion-en-seguridad-defensiva/main/02-fundamentos-python/images/resiliencia.png" width="420"/>

In [ ]:
import time, random

def reintentar(max_intentos=3, espera=1):
    def decorador(func):
        def wrapper(*args, **kwargs):
            for i in range(1, max_intentos + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"  intento {i} falló: {e}")
                    if i == max_intentos:
                        raise
                    time.sleep(espera)
        return wrapper
    return decorador

@reintentar(max_intentos=3, espera=0.5)
def consultar_api(ip):
    if random.random() < 0.6:          # simulamos fallos de red
        raise ConnectionError("timeout")
    return {"ip": ip, "malicioso": True}

print(consultar_api("10.0.0.5"))